# 01 - WM-811K 数据探索

本 notebook 用于探索 WM-811K 晶圆缺陷数据集，了解数据结构、类别分布和缺陷模式可视化。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

Matplotlib is building the font cache; this may take a moment.


## 1. 加载数据

In [2]:
# 加载原始数据（注意：文件较大，加载需要 10-30 秒）
DATA_PATH = '../data/raw/MIR-WM811K/Python/WM811K.pkl'
df = pd.read_pickle(DATA_PATH)

# 清洗：failureType 可能含 numpy.ndarray（不可哈希），统一转为可哈希类型
def _to_hashable(x):
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return str(x)
    return x

df['failureType'] = df['failureType'].apply(_to_hashable)
print(f'数据形状: {df.shape}')
print(f'列名: {list(df.columns)}')
df.head()

数据形状: (811457, 6)
列名: ['dieSize', 'failureType', 'lotName', 'trainTestLabel', 'waferIndex', 'waferMap']


,dieSize,failureType,lotName,trainTestLabel,waferIndex,waferMap
0,1683.0,none,lot1,Training,1.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,1683.0,none,lot1,Training,2.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,1683.0,none,lot1,Training,3.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,1683.0,none,lot1,Training,4.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,1683.0,none,lot1,Training,5.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


## 2. 基本信息

- `waferMap`: 晶圆图（numpy 数组，0=背景，1=正常 die，2=缺陷 die）
- `failureType`: 缺陷类型（None 表示无标签）
- `trainTestLabel`: 训练/测试划分
- `lotName`: 批次号
- `waferIndex`: 批次内晶圆序号
- `dieSize`: die 尺寸

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 811457 entries, 0 to 811456
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   dieSize         811457 non-null  float64
 1   failureType     811457 non-null  object 
 2   lotName         811457 non-null  object 
 3   trainTestLabel  811457 non-null  object 
 4   waferIndex      811457 non-null  float64
 5   waferMap        811457 non-null  object 
dtypes: float64(2), object(4)
memory usage: 37.1+ MB


## 3. 缺陷类型分布

In [4]:
# 用 Counter 快速统计（比 pandas value_counts 快）
failure_counts = Counter(df['failureType'].tolist())
print('=== 所有数据 failureType 分布 ===')
for k, v in sorted(failure_counts.items(), key=lambda x: -x[1]):
    print(f'  {str(k):15s}: {v:>7d} ({v/len(df)*100:.2f}%)')

TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
# 只看有标签的数据
labeled = df[df['failureType'].notna()].copy()
print(f'有标签数据: {len(labeled)} / {len(df)} ({len(labeled)/len(df)*100:.1f}%)')

labeled_counts = Counter(labeled['failureType'].tolist())
print('\n=== 有标签数据 failureType 分布 ===')
for k, v in sorted(labeled_counts.items(), key=lambda x: -x[1]):
    print(f'  {str(k):15s}: {v:>6d} ({v/len(labeled)*100:.2f}%)')

In [ ]:
# 可视化类别分布
types = sorted(labeled_counts.keys())
counts = [labeled_counts[t] for t in types]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(types, counts, color='steelblue')
ax.set_title('WM-811K Labeled Failure Type Distribution')
ax.set_xlabel('Failure Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/01_failure_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 训练/测试划分

In [ ]:
split_counts = Counter(df['trainTestLabel'].tolist())
print('=== trainTestLabel 分布 ===')
for k, v in sorted(split_counts.items(), key=lambda x: -x[1]):
    print(f'  {str(k):15s}: {v:>7d} ({v/len(df)*100:.2f}%)')

## 5. 可视化各类缺陷模式

In [ ]:
# 每种缺陷类型显示 3 个示例
failure_types = sorted(labeled['failureType'].unique())
n_types = len(failure_types)
n_examples = 3

fig, axes = plt.subplots(n_types, n_examples, figsize=(3*n_examples, 3*n_types))
if n_types == 1:
    axes = axes.reshape(1, -1)

for i, ftype in enumerate(failure_types):
    subset = labeled[labeled['failureType'] == ftype]
    sample = subset.sample(n=min(n_examples, len(subset)), random_state=42)
    for j, (_, row) in enumerate(sample.iterrows()):
        ax = axes[i, j]
        ax.imshow(row['waferMap'], cmap='gray')
        ax.set_title(f'{ftype}' if j == 0 else '', fontsize=11)
        ax.axis('off')
    # 空白位置
    for j in range(len(sample), n_examples):
        axes[i, j].axis('off')

plt.suptitle('Wafer Map Examples by Failure Type', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/01_wafer_map_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 晶圆图尺寸分析

In [ ]:
# 统计 waferMap 的形状分布
shapes = [wm.shape for wm in labeled['waferMap'].tolist()]
shape_counts = Counter(shapes)
print('=== waferMap 形状分布（前 10）===')
for shape, count in sorted(shape_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {str(shape):15s}: {count:>6d}')

print(f'\n总共有 {len(shape_counts)} 种不同的形状')
print(f'这意味着需要 resize 到统一尺寸才能输入神经网络')

## 7. 关键发现总结

1. **数据量**: 811,457 张晶圆图，其中约 172,950 张有标签
2. **类别不平衡**: 各类缺陷样本数差异很大，Normal 最多，部分类别很少
3. **尺寸不统一**: waferMap 有多种不同尺寸，需要 resize 到统一大小（如 64x64 或 128x128）
4. **像素值**: 0=背景，1=正常 die，2=缺陷 die
5. **下一步**: 数据预处理（resize、归一化）、划分训练/验证集、构建 baseline CNN